<a href="https://colab.research.google.com/github/xreemas7x-pixel/-IT362-Thmanyah-Podcast-Analysis/blob/main/Thmanyah_Podcast_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

These first cells document our initial attempt to find the correct Thmanyah podcast playlists. We kept them to show how we started the data collection phase they are not part of the final pipeline. The clean, working version is provided below under "Get the correct playlist IDs directly from Thmanyah's YouTube channel.

In [ ]:
import requests

CHANNEL_ID = "UCwjLh640nGXSGa9iHRS31ag"

url = "https://www.googleapis.com/youtube/v3/playlists"

params = {
    "part": "snippet,contentDetails",
    "channelId": CHANNEL_ID,
    "maxResults": 50,
    "key": API_KEY
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)
print(response.text)

Status Code: 200
{
  "kind": "youtube#playlistListResponse",
  "etag": "nbpRi7U9oR5qknLbugUOzGKXJ-c",
  "pageInfo": {
    "totalResults": 32,
    "resultsPerPage": 50
  },
  "items": [
    {
      "kind": "youtube#playlist",
      "etag": "jT3BC6a6Yo5WG4FwKpuFxqvCojs",
      "id": "PLeNNUmE-BlnEc8c8ZFqPbF4wJBjWnbaAE",
      "snippet": {
        "publishedAt": "2025-09-02T22:13:02.823262Z",
        "channelId": "UCwjLh640nGXSGa9iHRS31ag",
        "title": "بودكاست أرباح",
        "description": "بودكاست حواري عن سوق الأسهم. في كل حلقة يجتمع أنس الراجحي وسعيد عبدالجبار لنقاش مواضيع وأسئلة تساعدك على اتخاذ قرارات استثمارية أفضل.",
        "thumbnails": {
          "default": {
            "url": "https://i.ytimg.com/vi/4LT66LgD4Ec/default.jpg",
            "width": 120,
            "height": 90
          },
          "medium": {
            "url": "https://i.ytimg.com/vi/4LT66LgD4Ec/mqdefault.jpg",
            "width": 320,
            "height": 180
          },
          "high": {
      

In [ ]:
all_playlists = []
next_page_token = None

while True:

    params = {
        "part": "snippet,contentDetails",
        "channelId": CHANNEL_ID,
        "maxResults": 50,
        "key": API_KEY
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(url, params=params)
    data = response.json()

    for item in data["items"]:
        all_playlists.append({
            "playlist_title": item["snippet"]["title"],
            "playlist_id": item["id"],
            "video_count": item["contentDetails"]["itemCount"]
        })

    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        break

playlists_df = pd.DataFrame(all_playlists)

playlists_df

,playlist_title,playlist_id,video_count
0,بودكاست أرباح,PLeNNUmE-BlnEc8c8ZFqPbF4wJBjWnbaAE,23
1,بودكاست جادي,PLeNNUmE-BlnEqeZQVrv5LuWyCGp3EloX6,19
2,بودكاست ذا قال,PLeNNUmE-BlnGwvIudtsVFIuKb4N_ods18,31
3,بودكاست النشرة السينمائية,PLeNNUmE-BlnFXTKtkgaHgFxMRNButo9eb,20
4,الأكثر تأثيرًا في 2023,PLeNNUmE-BlnHOvl5QLUeMUyr4WgUeKdUB,10
5,لعيش حياة صحية,PLeNNUmE-BlnFxpCBC8kiNq0QuSQvm_2Sq,11
6,لفهم أعمق للمناطق والدول,PLeNNUmE-BlnH1GiFGfGD0qJ5e2g3lEFiO,18
7,بودكاست آدم,PLeNNUmE-BlnFmi6C5-aKc6NQ_uXmmA5TP,23
8,بودكاست السوق,PLeNNUmE-BlnG9JQSWJ3RHEqION3UL3ITk,43
9,بودكاست مربّع,PLeNNUmE-BlnEuww8BC6HTe-v_8P--aeSV,49


In [ ]:
PLAYLISTS = {
    row["playlist_title"]: row["playlist_id"]
    for _, row in selected_playlists.iterrows()
}

PLAYLISTS

{'بودكاست مرتدة': 'PLeNNUmE-BlnGmvn7CrbD30RSfNkeqgx22',
 'بودكاست سوالف بزنس': 'PLeNNUmE-BlnE5GVCM_aZtWCLzCzHYA-ho',
 'بودكاست سقراط': 'PLeNNUmE-BlnH6BfF8o6I3QQDCPYhwjBWZ',
 'بودكاست فنجان': 'PLeNNUmE-BlnEls2kOkiV2CXMezV6oaq8I'}

In [ ]:
PLAYLISTS = {}

for _, row in selected_playlists.iterrows():
    title = row["playlist_title"].strip()

    if "فنجان" in title:
        PLAYLISTS["Fanjan"] = row["playlist_id"]
    elif "سوالف بزنس" in title:
        PLAYLISTS["Swalif_Business"] = row["playlist_id"]
    elif "سقراط" in title:
        PLAYLISTS["Socrates"] = row["playlist_id"]
    elif "مرتدة" in title:
        PLAYLISTS["Mortada"] = row["playlist_id"]

PLAYLISTS

{'Mortada': 'PLeNNUmE-BlnGmvn7CrbD30RSfNkeqgx22',
 'Swalif_Business': 'PLeNNUmE-BlnE5GVCM_aZtWCLzCzHYA-ho',
 'Socrates': 'PLeNNUmE-BlnH6BfF8o6I3QQDCPYhwjBWZ',
 'Fanjan': 'PLeNNUmE-BlnEls2kOkiV2CXMezV6oaq8I'}

# IT362 Principles of Data Science
## Phase 1: Data Collection – Thmanyah Podcast Analysis

This notebook documents the data collection process for four Thmanyah podcast programs:

- Fanjan
- Swalif Business
- Socrates
- Mortada

The data were collected using the YouTube Data API v3.

The purpose of this notebook is to retrieve raw podcast video data, including metadata, video duration, thumbnails, and engagement statistics such as views, likes, and comments.

In [ ]:
import requests
import json
import pandas as pd

In [ ]:
# YouTube Data API key
API_KEY = "YOUR_API_KEY_HERE"

# Thmanyah YouTube Channel ID
CHANNEL_ID = "UCwjLh640nGXSGa9iHRS31ag"

In [ ]:
# Get the correct playlist IDs directly from Thmanyah's YouTube channel

playlist_url = "https://www.googleapis.com/youtube/v3/playlists"

all_playlists = []
next_page_token = None

while True:
    params = {
        "part": "snippet,contentDetails",
        "channelId": CHANNEL_ID,
        "maxResults": 50,
        "key": API_KEY
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(playlist_url, params=params)
    data = response.json()

    if response.status_code != 200:
        print("API Error:", data)
        break

    all_playlists.extend(data["items"])

    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        break


# Save the four required podcast playlists
PLAYLISTS = {}

for item in all_playlists:
    title = item["snippet"]["title"].strip()
    playlist_id = item["id"]

    if "فنجان" in title:
        PLAYLISTS["Fanjan"] = playlist_id

    elif "سوالف بزنس" in title:
        PLAYLISTS["Swalif_Business"] = playlist_id

    elif "سقراط" in title:
        PLAYLISTS["Socrates"] = playlist_id

    elif "مرتدة" in title:
        PLAYLISTS["Mortada"] = playlist_id


PLAYLISTS

{'Mortada': 'PLeNNUmE-BlnGmvn7CrbD30RSfNkeqgx22',
 'Swalif_Business': 'PLeNNUmE-BlnE5GVCM_aZtWCLzCzHYA-ho',
 'Socrates': 'PLeNNUmE-BlnH6BfF8o6I3QQDCPYhwjBWZ',
 'Fanjan': 'PLeNNUmE-BlnEls2kOkiV2CXMezV6oaq8I'}

In [ ]:
def get_video_ids(playlist_id):

    video_ids = []
    next_page_token = None

    while True:

        url = "https://www.googleapis.com/youtube/v3/playlistItems"

        params = {
            "part": "contentDetails",
            "playlistId": playlist_id,
            "maxResults": 50,
            "key": API_KEY
        }

        if next_page_token:
            params["pageToken"] = next_page_token

        response = requests.get(url, params=params)
        data = response.json()

        if response.status_code != 200:
            print("API Error:", data)
            return video_ids

        for item in data.get("items", []):
            video_ids.append(
                item["contentDetails"]["videoId"]
            )

        next_page_token = data.get("nextPageToken")

        if not next_page_token:
            break

    return video_ids

In [ ]:
fanjan_ids = get_video_ids(PLAYLISTS["Fanjan"])

print("Fanjan videos:", len(fanjan_ids))

Fanjan videos: 257


## Retrieve Video Details

After collecting all video IDs from each playlist, the YouTube Data API is used to retrieve detailed information about each video, including metadata, duration, and engagement statistics.

In [ ]:
def get_video_details(video_ids):
    all_video_details = []

    url = "https://www.googleapis.com/youtube/v3/videos"

    # YouTube API accepts up to 50 video IDs per request
    for i in range(0, len(video_ids), 50):

        batch_ids = video_ids[i:i + 50]

        params = {
            "part": "snippet,contentDetails,statistics",
            "id": ",".join(batch_ids),
            "key": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if response.status_code != 200:
            print("API Error:", data)
            break

        all_video_details.extend(data["items"])

    return all_video_details

In [ ]:
fanjan_details = get_video_details(fanjan_ids)

print("Fanjan detailed records:", len(fanjan_details))

Fanjan detailed records: 257


In [ ]:
fanjan_details[0]

{'kind': 'youtube#video',
 'etag': 'U07Kl_U1NXSdS63xC5PLsv9I9w4',
 'id': 'HIqF3wF7gxY',
 'snippet': {'publishedAt': '2026-03-04T11:00:33Z',
  'channelId': 'UCwjLh640nGXSGa9iHRS31ag',
  'title': 'لماذا خلق العباسيون فتنة خلق القرآن | بودكاست فنجان',
  'description': 'حدثت في العصر العباسي فتنةٌ عظيمة امتحنت المسلمين، وأراقت الدماء وقتلت كبار العلماء، وهي فتنة خلق القرآن التي ادَّعت أن القرآن مخلوق مُحدَث. هذه الفتنة، مع أنها امتحنت المسلمين في عقيدتهم، يرى ضيفي أنها كانت أكبر من مجرد قضية عقائدية، وإنما قضية وُظفت لأغراض سياسية أيضًا!\n\nيعود ضيفي بشار عواد، المؤرخ والباحث العراقي في هذه الحلقة إلى سياقات تاريخية طويلة لفهم هذه الفتنة، ويسرد القصة الكاملة لنشأتها وأصلها ونهايتها.\n\nبعد النزاع الشهير بين الخليفتين الأمين والمأمون، كان الأخير تبنَّى فكرة خلق القرآن، وجاهد في نشرها وعزل من لم يؤمن بها، غير أن ضيفي يشكّك أصلًا في اعتناق المأمون هذا المعتقد، حيث يقول إن شخصية المأمون معقدة ودراماتيكية، وكانت له مصالح معيّنة في نشره هذه الفتنة.\n\nوعلى مدى حكم المأمون، تأذَّى الكثير من كبار 

In [ ]:
all_podcast_data = {}

for podcast_name, playlist_id in PLAYLISTS.items():

    print(f"Collecting {podcast_name}...")

    video_ids = get_video_ids(playlist_id)
    video_details = get_video_details(video_ids)

    all_podcast_data[podcast_name] = video_details

    print(f"{podcast_name}: {len(video_details)} videos")

Mortada: 116 videos
Swalif_Business: 147 videos
Socrates: 181 videos
Fanjan: 257 videos


## Save Raw Data

The collected API responses are saved as raw JSON files for each podcast.
This preserves the original data before any preprocessing or transformation.

In [ ]:
for podcast_name, video_details in all_podcast_data.items():

    filename = f"{podcast_name}_raw.json"

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(
            video_details,
            file,
            ensure_ascii=False,
            indent=4
        )

    print(f"Saved: {filename}")

Saved: Mortada_raw.json
Saved: Swalif_Business_raw.json
Saved: Socrates_raw.json
Saved: Fanjan_raw.json


## Convert Raw Data to a Structured DataFrame

The raw YouTube API responses are converted into a structured tabular format.
Each row represents one podcast video, while the columns contain selected metadata,
content information, and engagement statistics.

In [ ]:
structured_data = []

for podcast_name, videos in all_podcast_data.items():

    for video in videos:

        snippet = video.get("snippet", {})
        content_details = video.get("contentDetails", {})
        statistics = video.get("statistics", {})

        structured_data.append({
            "podcast": podcast_name,
            "video_id": video.get("id"),
            "title": snippet.get("title"),
            "description": snippet.get("description"),
            "published_at": snippet.get("publishedAt"),
            "duration": content_details.get("duration"),
            "view_count": statistics.get("viewCount"),
            "like_count": statistics.get("likeCount"),
            "comment_count": statistics.get("commentCount"),
            "thumbnail_url": snippet.get("thumbnails", {})
                                    .get("high", {})
                                    .get("url")
        })

df = pd.DataFrame(structured_data)

df.head()

,podcast,video_id,title,description,published_at,duration,view_count,like_count,comment_count,thumbnail_url
0,Mortada,5GRMhsKyN8M,تشيلسي أمام الجميع بطلًا للعالم | بودكاست مرتدة,في الحلقة الأخيرة من بطولة كأس العالم للأندية،...,2025-07-15T03:00:09Z,PT1H47M31S,83059,1266,182,https://i.ytimg.com/vi/5GRMhsKyN8M/hqdefault.jpg
1,Mortada,azAJhdXaOfE,باريس يسحق الريال برباعية ويواجه تشيلسي في الن...,في الحلقة قبل الأخيرة من موسم كأس العالم للأند...,2025-07-11T03:00:03Z,PT1H52M3S,140916,2802,301,https://i.ytimg.com/vi/azAJhdXaOfE/hqdefault.jpg
2,Mortada,3vb9-ti6-R4,هل قدّم الهلال بطولة مشرّفة؟ | بودكاست مرتدة,في هذه الحلقة، نناقش أبرز أحداث دور الثمانية م...,2025-07-06T09:00:02Z,PT1H44M20S,97856,2033,212,https://i.ytimg.com/vi/3vb9-ti6-R4/hqdefault.jpg
3,Mortada,y8IqdgPuUbU,الهلال يقصي مان سيتي من المونديال برباعية | بو...,بعد فوز الهلال الكبير على مانشستر سيتي، أحد أق...,2025-07-02T08:58:20Z,PT1H43M22S,189738,4276,539,https://i.ytimg.com/vi/y8IqdgPuUbU/hqdefault.jpg
4,Mortada,XFTcez1REdU,أمل الهلال الوحيد في الفوز على السيتي | بودكاس...,في هذه الحلقة نناقش أبرز أحداث مباريات كأس الع...,2025-06-28T05:15:25Z,PT2H30S,169814,2906,298,https://i.ytimg.com/vi/XFTcez1REdU/hqdefault.jpg


In [ ]:
print("Total records:", len(df))
print("Number of columns:", len(df.columns))

df.info()

Total records: 701
Number of columns: 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 701 entries, 0 to 700
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   podcast        701 non-null    object
 1   video_id       701 non-null    object
 2   title          701 non-null    object
 3   description    701 non-null    object
 4   published_at   701 non-null    object
 5   duration       701 non-null    object
 6   view_count     701 non-null    object
 7   like_count     701 non-null    object
 8   comment_count  701 non-null    object
 9   thumbnail_url  701 non-null    object
dtypes: object(10)
memory usage: 54.9+ KB


## Initial Data Quality Inspection

Before preprocessing, the dataset is inspected for duplicate videos, missing values,
and the distribution of records across the four podcast programs.

In [ ]:
print("Duplicate Video IDs:", df["video_id"].duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nVideos per Podcast:")
print(df["podcast"].value_counts())

Duplicate Video IDs: 0

Missing Values:
podcast          0
video_id         0
title            0
description      0
published_at     0
duration         0
view_count       0
like_count       0
comment_count    0
thumbnail_url    0
dtype: int64

Videos per Podcast:
podcast
Fanjan             257
Socrates           181
Swalif_Business    147
Mortada            116
Name: count, dtype: int64


## Data Collection Summary

Data were collected from four Thmanyah podcast programs using the YouTube Data API v3:

- Fanjan: 257 videos
- Socrates: 181 videos
- Swalif Business: 147 videos
- Mortada: 116 videos

A total of 701 video records were collected.

For each video, the collected data include:
- Video ID
- Title
- Description
- Publication date
- Duration
- View count
- Like count
- Comment count
- Thumbnail URL

The initial inspection showed no duplicate video IDs and no missing values in the collected attributes.

The raw API responses were preserved as JSON files for each podcast program.